# 01 — Schwarzschild Basics

**Spacetime Lab — Phase 1 concept notebook**

This notebook walks through the physics of the Schwarzschild metric — the unique static, spherically symmetric vacuum solution to Einstein's field equations — and verifies every claim against the `spacetime_lab.metrics.Schwarzschild` implementation.

**What you will learn**

1. The Schwarzschild line element and what its coordinates mean
2. That Schwarzschild is a *vacuum* solution ($R_{\mu\nu} = 0$)
3. The difference between a coordinate singularity and a curvature singularity
4. Three critical radii: the event horizon ($r=2M$), the photon sphere ($r=3M$), and the ISCO ($r=6M$)
5. Tortoise and Kruskal–Szekeres coordinates, and why we ever introduce them
6. The effective potential for timelike and null geodesics
7. Horizon thermodynamics: surface gravity, Hawking temperature, Bekenstein–Hawking entropy

**References**

- Wald, *General Relativity*, ch. 6
- Misner–Thorne–Wheeler, *Gravitation*, ch. 31
- Carroll, *Spacetime and Geometry*, ch. 5

**Conventions**: metric signature $(-,+,+,+)$, geometric units $G = c = 1$, so mass, time and length share the same dimension.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from spacetime_lab.metrics import Schwarzschild

sp.init_printing(use_unicode=True)
plt.rcParams['figure.figsize'] = (7, 4.5)

## 1. The line element

In Schwarzschild coordinates $(t, r, \theta, \varphi)$ the line element is

$$
ds^2 = -\left(1 - \frac{2M}{r}\right) dt^2 + \left(1 - \frac{2M}{r}\right)^{-1} dr^2 + r^2\,(d\theta^2 + \sin^2\theta\, d\varphi^2)
$$

**Coordinates are not what they seem.**

- $t$ is the time measured by a *stationary observer at infinity*, not by anyone near the black hole.
- $r$ is a *circumferential* (or 'areal') radius: the sphere at radius $r$ has area $4\pi r^2$. It is **not** a proper distance to the origin.
- $(\theta, \varphi)$ are standard angular coordinates on each sphere of symmetry.

The factor $f(r) = 1 - 2M/r$ is the central object: it is negative inside $r=2M$, zero on it, and positive outside. At $r = 2M$ the coefficient $1/f$ blows up — but as we will see, **the curvature stays finite there**.

In [ ]:
# Symbolic instance — mass is a symbol M
bh = Schwarzschild()
print(bh)
bh.metric_tensor

In [ ]:
# Human-readable line element
print(bh.line_element_latex())

## 2. Schwarzschild is a vacuum solution

The Einstein field equations are

$$
R_{\mu\nu} - \tfrac{1}{2} R\, g_{\mu\nu} = 8\pi T_{\mu\nu}.
$$

'Vacuum' means $T_{\mu\nu} = 0$, which is equivalent to $R_{\mu\nu} = 0$ (and hence $R=0$). Schwarzschild is vacuum *outside* the central mass: there is no matter field anywhere in the spacetime the metric describes.

Our base class computes the Ricci tensor from the metric via Christoffels → Riemann → contraction. If the implementation is correct, the Ricci scalar must be exactly zero for Schwarzschild. This is the most basic consistency check.

In [ ]:
R_scalar = bh.ricci_scalar()
print('Ricci scalar:', R_scalar)
assert sp.simplify(R_scalar) == 0
print('PASS: Schwarzschild satisfies the vacuum Einstein equations.')

## 3. Coordinate singularity vs curvature singularity

The metric components $g_{tt}$ and $g_{rr}$ blow up at $r = 0$ and $r = 2M$. Is either of these a real, physical singularity?

The Ricci scalar is identically zero and cannot distinguish them. The **Kretschmann scalar**

$$
K \equiv R_{\mu\nu\rho\sigma} R^{\mu\nu\rho\sigma} = \frac{48 M^2}{r^6}
$$

*can*. It is a true tensorial invariant: if $K$ blows up, no coordinate change can hide it.

- At $r = 2M$: $K = 48 M^2 / (2M)^6 = 3/(4M^4)$ — **finite**. The horizon is just a coordinate singularity.
- At $r \to 0$: $K \to \infty$. This is a **real curvature singularity**: tidal forces diverge, no observer survives.

In [ ]:
r = sp.Symbol('r', positive=True)
print('K(r) =', bh.kretschmann_scalar(r))
print('K(2M) =', sp.simplify(bh.kretschmann_scalar(2 * bh.mass)))
print('K(0+) ->', sp.limit(bh.kretschmann_scalar(r), r, 0, '+'))

In [ ]:
# Plot K(r) for M=1 on a log scale
bh_num = Schwarzschild(mass=1.0)
rs = np.linspace(0.2, 10, 400)
Ks = [float(bh_num.kretschmann_scalar(ri)) for ri in rs]

fig, ax = plt.subplots()
ax.plot(rs, Ks, color='#0b6fc2')
ax.axvline(2.0, color='crimson', ls='--', label='event horizon r=2M')
ax.axvline(3.0, color='goldenrod', ls=':', label='photon sphere r=3M')
ax.axvline(6.0, color='seagreen', ls=':', label='ISCO r=6M')
ax.set_yscale('log')
ax.set_xlabel('r / M')
ax.set_ylabel(r'$K(r) = R_{abcd}R^{abcd}$')
ax.set_title('Kretschmann scalar: finite at horizon, diverges at r=0')
ax.legend()
plt.tight_layout()
plt.show()

## 4. The three critical radii

For Schwarzschild with mass $M$:

| Radius | Value | Meaning |
|---|---|---|
| **Event horizon** | $r_s = 2M$ | Null one-way membrane. Nothing inside can reach $r > 2M$ in the future. |
| **Photon sphere** | $r_\text{ph} = 3M$ | Unstable circular null orbit. Photons with exactly this impact parameter loop around. |
| **ISCO** | $r_\text{ISCO} = 6M$ | Smallest stable circular orbit for a massive test particle. Inside this, any perturbation sends you inward. |

Derivations (sketch): the event horizon is the locus where $g_{tt} = 0$ for the Killing vector $\partial_t$. The photon sphere and ISCO come from the effective potential — we will see them explicitly in section 6.

In [ ]:
bh1 = Schwarzschild(mass=1.0)
print(f'event horizon r_s      = {bh1.event_horizon()}')
print(f'photon sphere r_ph     = {bh1.photon_sphere()}')
print(f'ISCO          r_ISCO   = {bh1.isco()}')

## 5. The tortoise coordinate $r^*$

Schwarzschild coordinates are terrible for describing *wave propagation* near the horizon — the coefficient of $dr^2$ blows up and any radial wave equation acquires a nasty pole. The tortoise coordinate

$$
r^*(r) = r + 2M \ln\left|\frac{r}{2M} - 1\right|
$$

'stretches' the horizon to $r^* \to -\infty$. In $(t, r^*)$ the 2D radial wave equation for a scalar field takes the flat form $(-\partial_t^2 + \partial_{r^*}^2)\phi = V(r)\phi$, and ingoing / outgoing null rays travel on lines $t \pm r^* = \text{const}$. You will meet $r^*$ again in Phase 5 when we compute quasinormal modes.

It diverges logarithmically near the horizon: slowly enough that nothing reaches $r = 2M$ in finite Schwarzschild time $t$, fast enough that a free-falling observer *does* cross in finite proper time.

In [ ]:
bh1 = Schwarzschild(mass=1.0)
rs_plot = np.linspace(2.01, 10, 400)
r_stars = np.array([bh1.tortoise_coordinate(ri) for ri in rs_plot])

fig, ax = plt.subplots()
ax.plot(rs_plot, r_stars, color='#0b6fc2')
ax.axvline(2.0, color='crimson', ls='--', label='horizon r=2M')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('r / M')
ax.set_ylabel(r'$r^* / M$')
ax.set_title('Tortoise coordinate: horizon pushed to -infinity')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Effective potential and geodesics

Because Schwarzschild is static and spherically symmetric, two constants of motion exist for every geodesic: the conserved energy $E = -g_{tt}\dot t = (1 - 2M/r)\dot t$ and the angular momentum $L = g_{\varphi\varphi}\dot\varphi = r^2 \dot\varphi$ (on the equator $\theta = \pi/2$).

Eliminating these reduces radial motion to a 1D energy equation

$$
\left(\frac{dr}{d\tau}\right)^2 = E^2 - V_\text{eff}(r),
$$

with

$$
V_\text{eff}^\text{massive}(r) = \left(1 - \frac{2M}{r}\right)\left(1 + \frac{L^2}{r^2}\right), \qquad V_\text{eff}^\text{photon}(r) = \left(1 - \frac{2M}{r}\right)\frac{L^2}{r^2}.
$$

The *maximum* of $V_\text{eff}^\text{photon}$ is at $r = 3M$ — this is why the photon sphere is there. Circular timelike orbits are extrema of $V_\text{eff}^\text{massive}$, and the ISCO is the smallest stable one (where the extremum becomes an inflection, at $L = \sqrt{12}\,M$).

In [ ]:
bh1 = Schwarzschild(mass=1.0)
rs_plot = np.linspace(2.01, 15, 600)

fig, ax = plt.subplots()
# Massive: a few L values
for L in [3.0, np.sqrt(12), 4.5]:
    V = np.array([float(bh1.effective_potential(ri, L, 'massive')) for ri in rs_plot])
    label = f'massive, L={L:.2f}'
    if abs(L - np.sqrt(12)) < 1e-6:
        label += ' (ISCO)'
    ax.plot(rs_plot, V, label=label)

# Photon
Vp = np.array([float(bh1.effective_potential(ri, 1.0, 'photon')) for ri in rs_plot])
ax.plot(rs_plot, Vp, color='k', ls='--', label='photon, L=1')

ax.axvline(2.0, color='crimson', ls=':', lw=0.8)
ax.axvline(3.0, color='goldenrod', ls=':', lw=0.8)
ax.axvline(6.0, color='seagreen', ls=':', lw=0.8)
ax.set_xlabel('r / M')
ax.set_ylabel(r'$V_\mathrm{eff}(r)$')
ax.set_title('Effective potential for radial motion')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Direct check: V_eff^photon is maximised at r=3M
for r_test in [2.5, 2.8, 3.0, 3.5, 4.0]:
    V = float(bh1.effective_potential(r_test, 1.0, 'photon'))
    print(f'V_eff^photon({r_test:.1f}M) = {V:.6f}')

## 7. Kruskal–Szekeres: the maximal extension

Schwarzschild coordinates only cover the exterior $r > 2M$. Kruskal–Szekeres coordinates $(T, X)$ cover the maximally extended spacetime — four regions linked by the horizon. In Region I (our universe, $r > 2M$):

$$
T = \sqrt{\tfrac{r}{2M} - 1}\, e^{r/4M} \sinh\!\left(\tfrac{t}{4M}\right), \qquad X = \sqrt{\tfrac{r}{2M} - 1}\, e^{r/4M} \cosh\!\left(\tfrac{t}{4M}\right).
$$

Lines of constant $r$ become *hyperbolae* $X^2 - T^2 = \text{const}$; lines of constant $t$ become *straight lines through the origin*. The horizon $r = 2M$ is the pair of null lines $T = \pm X$.

This coordinate system will be the starting point for Phase 2 — we will conformally compactify it to obtain the Schwarzschild Penrose diagram.

In [ ]:
bh1 = Schwarzschild(mass=1.0)

fig, ax = plt.subplots(figsize=(6, 6))

# Constant-r hyperbolae in Region I (r > 2M)
ts = np.linspace(-8, 8, 300)
for r_const in [2.2, 2.5, 3.0, 4.0, 6.0]:
    Ts = []
    Xs = []
    for t in ts:
        T, X = bh1.kruskal_coordinates(t, r_const, region=1)
        Ts.append(T); Xs.append(X)
    ax.plot(Xs, Ts, label=f'r={r_const}M')

# Horizon lines: T = +/- X
lim = 15
ax.plot([-lim, lim], [-lim, lim], 'k--', lw=1.0)
ax.plot([-lim, lim], [lim, -lim], 'k--', lw=1.0)

ax.set_xlim(-1, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel('X'); ax.set_ylabel('T')
ax.set_title('Kruskal–Szekeres: constant-r hyperbolae (Region I)')
ax.set_aspect('equal')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Horizon thermodynamics (preview)

A miraculous feature of black holes: the horizon obeys laws that look exactly like the laws of thermodynamics. For Schwarzschild:

- **Surface gravity** $\kappa = \dfrac{1}{4M}$ — the 'force per unit mass at infinity' required to hold a test particle at the horizon.
- **Hawking temperature** $T_H = \dfrac{\kappa}{2\pi} = \dfrac{1}{8\pi M}$ — the temperature of the thermal radiation an inertial observer at infinity detects.
- **Bekenstein–Hawking entropy** $S_{BH} = \dfrac{A}{4} = 4\pi M^2$ — one quarter of the horizon area.

These will become central later in Phases 4 (Hawking radiation) and 7–9 (holographic entanglement entropy).

In [ ]:
for M in [0.5, 1.0, 5.0, 10.0]:
    bh = Schwarzschild(mass=M)
    print(f'M = {M:5.1f}  |  kappa = {float(bh.surface_gravity()):.4f}  '
          f'|  T_H = {float(bh.hawking_temperature()):.5f}  '
          f'|  S_BH = {float(bh.bekenstein_hawking_entropy()):.3f}')

## 9. Closing checks (Phase 1 gate)

Before we advance to Phase 2 (causal structure and Penrose diagrams), the following claims must hold. The assertions below fail loudly if anything regresses.

In [ ]:
import math

bh_sym = Schwarzschild()
bh1 = Schwarzschild(mass=1.0)

# 1. Vacuum Einstein equations
assert sp.simplify(bh_sym.ricci_scalar()) == 0

# 2. Kretschmann is finite at horizon, diverges at 0
K_h = float(bh1.kretschmann_scalar(2.0))
K_0 = float(bh1.kretschmann_scalar(0.01))
assert math.isfinite(K_h)
assert K_0 > 1e10 * K_h

# 3. Three critical radii
assert bh1.event_horizon() == 2.0
assert bh1.photon_sphere() == 3.0
assert bh1.isco() == 6.0

# 4. Tortoise finite outside, diverges at horizon
assert abs(bh1.tortoise_coordinate(4.0) - 4.0) < 1e-10
assert bh1.tortoise_coordinate(2.01) < -5

# 5. Photon effective potential maximised at r=3M
V3 = float(bh1.effective_potential(3.0, 1.0, 'photon'))
V25 = float(bh1.effective_potential(2.5, 1.0, 'photon'))
V4 = float(bh1.effective_potential(4.0, 1.0, 'photon'))
assert V3 > V25 and V3 > V4

# 6. Kruskal Region I at t=0 satisfies T=0, X>0
T, X = bh1.kruskal_coordinates(0.0, 3.0, region=1)
assert abs(T) < 1e-12 and X > 0

# 7. Christoffel consistency (explicit formulas == base-class symbolic)
assert bh_sym.verify_christoffel_symbols()

# 8. Thermodynamic relations
assert abs(float(bh1.surface_gravity()) - 0.25) < 1e-12
assert abs(float(bh1.hawking_temperature()) - 1 / (8 * math.pi)) < 1e-12
assert abs(float(bh1.bekenstein_hawking_entropy()) - 4 * math.pi) < 1e-12

print('ALL PHASE 1 GATE CHECKS PASSED')

---

## What's next — Phase 2 teaser

In Phase 2 we will build the `spacetime_lab.diagrams` module. Our first goal: take the Kruskal–Szekeres plot above and *conformally compactify* it, mapping the infinite spacetime onto a finite diamond so the causal structure becomes visible at a glance. This is the **Penrose diagram** of the maximally extended Schwarzschild spacetime, with its characteristic interior, white-hole, and second-asymptotic regions.

Key concepts to master before touching code (Phase 2 concept gate):

1. Conformal transformations of the metric and why they preserve causal structure
2. The Minkowski Penrose diagram and its infinities $i^{\pm}, i^{0}, \mathscr{I}^{\pm}$
3. Advanced/retarded Eddington–Finkelstein coordinates
4. Kruskal → compactified coordinates via $\arctan$ of null Kruskal variables